In [20]:
import os, sys, importlib

root = os.getcwd()
while not os.path.isdir(os.path.join(root, "src")) and os.path.dirname(root) != root:
    root = os.path.dirname(root)
os.chdir(root)

sys.path = [p for p in sys.path if p not in ("src", "../src")]
src = os.path.join(root, "src")
if src not in sys.path:
    sys.path.insert(0, src)
importlib.invalidate_caches()

%load_ext autoreload
%autoreload 2

print("Project root:", root)
print("parsers.py found:", os.path.exists(os.path.join(src, "parsers.py")))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Project root: d:\shalini code\Projects\ai_lab
parsers.py found: True


In [2]:
import pandas as pd

with open("data/raw/OpenSSH_2k.log") as f:
    lines = f.read().splitlines()

df = pd.DataFrame({"line": lines})
print(len(df))
df.head(10)

2000


,line
0,Dec 10 06:55:46 LabSZ sshd[24200]: reverse map...
1,Dec 10 06:55:46 LabSZ sshd[24200]: Invalid use...
2,Dec 10 06:55:46 LabSZ sshd[24200]: input_usera...
3,Dec 10 06:55:46 LabSZ sshd[24200]: pam_unix(ss...
4,Dec 10 06:55:46 LabSZ sshd[24200]: pam_unix(ss...
5,Dec 10 06:55:48 LabSZ sshd[24200]: Failed pass...
6,Dec 10 06:55:48 LabSZ sshd[24200]: Connection ...
7,Dec 10 07:02:47 LabSZ sshd[24203]: Connection ...
8,Dec 10 07:07:38 LabSZ sshd[24206]: Invalid use...
9,Dec 10 07:07:38 LabSZ sshd[24206]: input_usera...


In [3]:
patterns = ["Failed password", "Invalid user", "Accepted password", "Connection closed"]
for p in patterns:
    print(p, df["line"].str.contains(p).sum())

Failed password 520
Invalid user 113
Accepted password 1
Connection closed 34


In [4]:
failed = df[df["line"].str.contains("Failed password")]
ips = failed["line"].str.extract(r"from (\d+\.\d+\.\d+\.\d+)")
ips[0].value_counts().head(10)

0
183.62.140.253     286
187.141.143.180     80
103.99.0.122        46
112.95.230.3        26
5.188.10.180        18
185.190.58.151      17
123.235.32.19        7
119.4.203.64         6
52.80.34.196         5
60.2.12.12           5
Name: count, dtype: int64

In [22]:
from parsers import parse_file

df = parse_file("data/raw/SSH.log")
print(len(df))
df["event_type"].value_counts()

280939


event_type
failed_password        177735
connection_closed       68958
failed_invalid_user     19659
invalid_user            14405
accepted_password         182
Name: count, dtype: int64

In [6]:
import sys; sys.path.append("../src")
from parsers import parse_file
from features import build_features
from detectors import detect_rules

df = parse_file("data/raw/OpenSSH_2k.log")
print(len(df)); print(df["event_type"].value_counts())

feats = build_features(df)
print(feats.describe())          

alerts = detect_rules(feats)
print(len(alerts))
alerts.head(10)

666
event_type
failed_password        385
failed_invalid_user    134
invalid_user           112
connection_closed       34
accepted_password        1
Name: count, dtype: int64
                           window      failed    invalid   accepted  \
count                          49   49.000000  49.000000  49.000000   
mean   2015-12-10 09:00:36.734694   10.591837   2.285714   0.020408   
min           2015-12-10 06:55:00    0.000000   0.000000   0.000000   
25%           2015-12-10 07:55:00    1.000000   0.000000   0.000000   
50%           2015-12-10 09:10:00    1.000000   1.000000   0.000000   
75%           2015-12-10 10:10:00    6.000000   1.000000   0.000000   
max           2015-12-10 11:00:00  141.000000  28.000000   1.000000   
std                           NaN   27.834270   5.291503   0.142857   

       unique_users      events  
count     49.000000   49.000000  
mean       2.448980   13.591837  
min        0.000000    1.000000  
25%        1.000000    2.000000  
50%        1.0

,alert_id,source_ip,window_start,failed_logins,unique_usernames,successful_logins,detector
0,A-001,103.99.0.122,2015-12-10 09:10:00,30,19,0,brute_force
1,A-002,103.99.0.122,2015-12-10 09:10:00,30,19,0,user_enumeration
2,A-003,103.99.0.122,2015-12-10 11:00:00,16,12,0,brute_force
3,A-004,103.99.0.122,2015-12-10 11:00:00,16,12,0,user_enumeration
4,A-005,112.95.230.3,2015-12-10 07:25:00,26,3,0,brute_force
5,A-006,183.62.140.253,2015-12-10 10:50:00,16,3,0,brute_force
6,A-007,183.62.140.253,2015-12-10 10:55:00,141,8,0,brute_force
7,A-008,183.62.140.253,2015-12-10 10:55:00,141,8,0,user_enumeration
8,A-009,183.62.140.253,2015-12-10 11:00:00,129,1,0,brute_force
9,A-010,185.190.58.151,2015-12-10 09:10:00,11,2,0,brute_force


In [7]:
df.to_csv("data/processed/ssh_parsed.csv", index=False)
alerts.to_json("data/processed/alerts.json", orient="records", indent=2)

In [23]:
import sys; sys.path.insert(0, "src")
from parsers import parse_file
from inject_attacks import inject
from features import build_features
from detectors import detect_rules
from evaluate import evaluate

df = parse_file("data/raw/SSH.log")
df2, truth = inject(df)
truth.to_csv("data/ground_truth.csv", index=False)
print("Events before/after injection:", len(df), "->", len(df2))
print(truth[["source_ip", "attack_type", "label", "start"]].to_string(index=False))

alerts = detect_rules(build_features(df2))
metrics, rows = evaluate(alerts, truth)
print("\nPer-case result:")
for ip, kind, res in rows:
    print(f"  {ip:<14} {kind:<24} {res}")
print("\nMetrics:", metrics)


import pandas as pd
from detectors import detect_iforest
feats = build_features(df2)
ml = detect_iforest(feats)
m_ml, rows_ml = evaluate(ml, truth, any_detector=True)
combined = pd.concat([alerts, ml], ignore_index=True)
m_c, rows_c = evaluate(combined, truth, any_detector=True)
print("\nRules only        :", metrics)
print("Isolation Forest  :", m_ml)
print("Combined          :", m_c)
print("Isolation Forest flagged", len(ml), "of", len(feats), "windows")
for ip, kind, res in rows_ml:
    print(f"  IF  {ip:<14} {kind:<24} {res}")

Events before/after injection: 280939 -> 281028
    source_ip            attack_type  label                         start
 203.0.113.10            brute_force attack 2015-12-16 00:25:08.104340462
 203.0.113.11       slow_brute_force attack 2016-01-01 06:31:17.487217648
 203.0.113.12       user_enumeration attack 2016-01-03 16:30:30.698140148
 203.0.113.13 success_after_failures attack 2015-12-28 21:15:33.406810970
198.51.100.20        typo_then_login benign 2015-12-22 18:10:22.000000000
198.51.100.21         single_failure benign 2015-12-30 02:40:10.000000000
198.51.100.22           normal_login benign 2015-12-12 17:35:55.000000000
198.51.100.23        forgot_password benign 2015-12-22 14:15:46.000000000

Per-case result:
  203.0.113.10   brute_force              detected
  203.0.113.11   slow_brute_force         MISSED
  203.0.113.12   user_enumeration         detected
  203.0.113.13   success_after_failures   detected
  198.51.100.20  typo_then_login          ok (no alert)
  198.51.1

In [24]:
print("Rules only       :", metrics)
print("Isolation Forest :", m_ml)
print("Combined         :", m_c)
print("IF flagged", len(ml), "of", len(feats), "windows")
print("\nIF per case:")
for ip, kind, res in rows_ml:
    print(" ", ip, kind, res)
print("\nCombined per case:")
for ip, kind, res in rows_c:
    print(" ", ip, kind, res)

Rules only       : {'TP': 3, 'FP': 0, 'FN': 1, 'precision': 1.0, 'recall': 0.75, 'f1': 0.86}
Isolation Forest : {'TP': 1, 'FP': 1, 'FN': 3, 'precision': 0.5, 'recall': 0.25, 'f1': 0.33}
Combined         : {'TP': 3, 'FP': 1, 'FN': 1, 'precision': 0.75, 'recall': 0.75, 'f1': 0.75}
IF flagged 367 of 7578 windows

IF per case:
  203.0.113.10 brute_force MISSED
  203.0.113.11 slow_brute_force MISSED
  203.0.113.12 user_enumeration detected
  203.0.113.13 success_after_failures MISSED
  198.51.100.20 typo_then_login ok (no alert)
  198.51.100.21 single_failure ok (no alert)
  198.51.100.22 normal_login ok (no alert)
  198.51.100.23 forgot_password false alarm

Combined per case:
  203.0.113.10 brute_force detected
  203.0.113.11 slow_brute_force MISSED
  203.0.113.12 user_enumeration detected
  203.0.113.13 success_after_failures detected
  198.51.100.20 typo_then_login ok (no alert)
  198.51.100.21 single_failure ok (no alert)
  198.51.100.22 normal_login ok (no alert)
  198.51.100.23 forgo